# 02 - Pluto+ inter-channel phase coherence

**Question:** what is the maximum coherent processing interval (CPI) usable in
passive-radar cross-ambiguity processing, i.e. the longest window for which
std(phi12) < 10 deg between the two Pluto+ RX channels?

**Physical setup (no transmitter involved):** ANT500 telescopic antenna
(extended for FM band) -> 2-way 50 ohm SMA power splitter -> two same-length
SMA cables -> RX1 and RX2. Target signal: the strongest local FM broadcast
station (88-108 MHz). FM carriers are constant-envelope, ideal for phase
tracking. Zero transmissions from us.

Both channels run at the same center frequency and sample rate, with AGC
disabled (manual gain, identical values). AGC would invalidate the
measurement.

**Safety:** the Pluto+ TX channels must remain permanently disabled. This is
a receive-only project (EU legal constraint for private operators). Nothing
in this notebook enables or touches TX.

With `RUN_CAPTURE = False` (default) the notebook executes top-to-bottom on
synthetic fallback data: no hardware, no pyadi-iio. Synthetic numbers only
demonstrate the pipeline; they are NOT measurements.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tacet.dsp import coherence
from tacet.loaders import cs16
from tacet.loaders import manifest as manifest_mod

# --- Configuration --------------------------------------------------------
RUN_CAPTURE = False            # True only at the bench, with the Pluto+ attached
SYNTHETIC_FALLBACK = True      # demo the pipeline with synthetic captures
STATION_FREQ_HZ = 98_500_000.0  # set after the FM band scan (placeholder)
CENTER_FREQ_HZ = 98_500_000.0   # both channels tuned here
SAMPLE_RATE = 2_560_000.0      # 2.56 Msps
RX_GAIN_DB = 30.0              # manual mode, identical on both channels
BW_HZ = 3_000.0                # carrier extraction bandwidth
FS_OUT = 5_000.0               # decimated rate for phase analysis
CHUNKS = 10                    # number of 60 s chunks
CHUNK_S = 60.0
LONG_RUN_S = 600.0
WINDOWS_S = (0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0)  # report table uses the first five
THRESHOLD_DEG = 10.0
DATA_DIR = Path("data/coherence_cal")
CAL_PATH = Path("data/calibration/pluto_plus.json")
print("config ok")


## Acquisition (hardware required, RX-only)

Cells in this section are inert unless `RUN_CAPTURE = True`. TX stays
permanently disabled: only RX channels are ever enabled.

First a quick scan of the FM band to find the strongest local station, then
the capture plan: 10 x 60 s chunks plus one continuous 600 s run, both
channels from the same buffer, saved as cs16 pairs under
`data/coherence_cal/<session>/` and indexed in the manifest.


In [ ]:
if RUN_CAPTURE:
    try:
        import adi
    except ImportError as exc:  # pragma: no cover - bench only
        raise RuntimeError(
            "pyadi-iio not installed. Run: uv sync --extra acquisition"
        ) from exc

    URI = "ip:192.168.2.1"  # Pluto+ over Ethernet; adjust as needed

    def open_pluto(center_hz, gain_db, sample_rate=SAMPLE_RATE):
        """Open the Pluto+ with BOTH RX channels from one device.

        TX must remain permanently disabled (receive-only project): this
        helper only ever reads/configures RX-side attributes.
        """
        sdr = adi.ad9361(uri=URI)
        sdr.sample_rate = int(sample_rate)
        sdr.rx_lo = int(center_hz)
        for ch in (0, 1):
            setattr(sdr, f"gain_control_mode_chan{ch}", "manual")
            setattr(sdr, f"hardwaregain_chan{ch}", int(gain_db))
        sdr.rx_enabled_channels = [0, 1]  # RX only; TX is never enabled
        return sdr

    def fm_band_scan(sdr, start_mhz=88.0, stop_mhz=108.0, step_khz=200):
        """RX-only scan: return the frequency of the strongest station."""
        powers = []
        freqs_mhz = np.arange(start_mhz, stop_mhz, step_khz / 1e3)
        for f in freqs_mhz:
            sdr.rx_lo = int(f * 1e6)
            buf = sdr.rx()
            a = np.asarray(buf[0]) if isinstance(buf, list) else np.asarray(buf)[:, 0]
            powers.append(float(np.mean(np.abs(a) ** 2)))
        best = int(np.argmax(powers))
        print(f"strongest station: {freqs_mhz[best]:.3f} MHz "
              f"(rel power {powers[best]:.3e})")
        return float(freqs_mhz[best] * 1e6)

    print("acquisition helpers defined (RUN_CAPTURE is on)")
else:
    print("acquisition skipped (RUN_CAPTURE = False)")


In [ ]:
if RUN_CAPTURE:  # pragma: no cover - bench only
    from datetime import datetime, timezone

    sdr = open_pluto(CENTER_FREQ_HZ, RX_GAIN_DB)
    STATION_FREQ_HZ = fm_band_scan(sdr)
    sdr.rx_lo = int(CENTER_FREQ_HZ)

    session = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    session_dir = DATA_DIR / session
    session_dir.mkdir(parents=True, exist_ok=True)

    def save_pair(z1, z2, path_a, path_b):
        for z, p in ((z1, path_a), (z2, path_b)):
            z = np.asarray(z)
            np.asarray([z.real, z.imag]).T.astype("<i2").tofile(p)

    plan = [(f"chunk_{i:02d}", CHUNK_S) for i in range(CHUNKS)]
    plan.append(("long_run", LONG_RUN_S))

    for name, dur_s in plan:
        n_total = int(dur_s * SAMPLE_RATE)
        n_per_buf = int(SAMPLE_RATE)  # ~1 s buffers
        acc1, acc2 = [], []
        got = 0
        while got < n_total:
            buf = sdr.rx()
            if isinstance(buf, list):
                a, b = np.asarray(buf[0]), np.asarray(buf[1])
            else:
                a, b = np.asarray(buf)[:, 0], np.asarray(buf)[:, 1]
            take = min(a.size, n_total - got)
            acc1.append(a[:take]); acc2.append(b[:take]); got += take
        pa = session_dir / f"chA_{name}.cs16"
        pb = session_dir / f"chB_{name}.cs16"
        save_pair(np.concatenate(acc1), np.concatenate(acc2), pa, pb)
        entry = {
            "id": f"{session}_{name}",
            "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "device": "pluto_plus", "protocol": "fm_carrier",
            "purpose": "coherence_cal",
            "center_freq_hz": float(CENTER_FREQ_HZ),
            "sample_rate_sps": float(SAMPLE_RATE),
            "format": "cs16", "duration_s": float(dur_s),
            "channels": [
                {"path": str(pa), "sha256": manifest_mod.compute_sha256(pa)},
                {"path": str(pb), "sha256": manifest_mod.compute_sha256(pb)},
            ],
            "antenna": "ANT500 telescopic + 2-way 50R SMA splitter",
            "operator_notes": f"station {STATION_FREQ_HZ/1e6:.3f} MHz",
            "gain": {"mode": "manual", "rx1_db": RX_GAIN_DB, "rx2_db": RX_GAIN_DB},
            "rf_chain": {"splitter": "2-way 50R SMA", "cable_len_m": 1.0},
        }
        manifest_mod.append_entry(entry)
        print(f"saved {name} ({dur_s:.0f} s)")
else:
    print("capture skipped (RUN_CAPTURE = False)")


## Loading

Real captures come from the manifest (`purpose = coherence_cal`). With no
recordings present and `SYNTHETIC_FALLBACK = True`, synthetic pairs stand in
so the analysis runs end to end. Synthetic results are placeholders, never
measurements.


In [ ]:
captures = []

rows = manifest_mod.query(purpose="coherence_cal") if Path("manifest.json").exists() else []
for e in rows:
    f_offset = 0.0  # both channels tuned to the station; offset kept explicit
    z1 = cs16.load_cs16(e["channels"][0]["path"])
    z2 = cs16.load_cs16(e["channels"][1]["path"])
    captures.append({"id": e["id"], "fs": e["sample_rate_sps"], "z1": z1,
                     "z2": z2, "f_offset": f_offset, "synthetic": False})

if not captures and SYNTHETIC_FALLBACK:
    rng_fs = 100_000.0
    f0 = 20_000.0
    for name, dur in [("synth_chunk_0", 30.0), ("synth_chunk_1", 30.0),
                      ("synth_long_run", 120.0)]:
        z1, z2 = coherence.synthetic_tone_pair(
            fs=rng_fs, duration_s=dur, f0=f0, phase_offset_deg=17.0,
            drift_deg_per_min=0.8, noise_sigma=2e-3, lag_samples=0)
        captures.append({"id": name, "fs": rng_fs, "z1": z1, "z2": z2,
                         "f_offset": f0, "synthetic": True})
    print("SYNTHETIC fallback captures in use - NOT measurements")
print(f"captures loaded: {[c['id'] for c in captures]}")


## Analysis

Per capture: extract the carrier (downconvert + narrowband filter + decimate),
unwrap the inter-channel phase difference, then window stats, drift rate and
the overlapping Allan deviation.


In [ ]:
results = {}
for cap in captures:
    fs = cap["fs"]
    fs_phi = fs / int(round(fs / FS_OUT))
    block = max(int(fs * 10), 1)
    b1 = coherence.downconvert_decimate(cap["z1"], fs, cap["f_offset"],
                                        bw_hz=BW_HZ, fs_out=FS_OUT,
                                        block_samples=block)
    b2 = coherence.downconvert_decimate(cap["z2"], fs, cap["f_offset"],
                                        bw_hz=BW_HZ, fs_out=FS_OUT,
                                        block_samples=block)
    phi12 = coherence.phase_difference(b1, b2)
    stats = coherence.window_phase_stats(phi12, fs_phi, windows_s=WINDOWS_S)
    drift, drift_err = coherence.drift_rate(phi12, fs_phi)
    taus, adev = coherence.allan_deviation(phi12, fs_phi)
    results[cap["id"]] = {
        "fs_phi": fs_phi, "phi12": phi12, "stats": stats,
        "drift_deg_min": drift, "drift_err_deg_min": drift_err,
        "taus": np.asarray(taus), "adev": np.asarray(adev),
        "synthetic": cap["synthetic"],
    }
    print(f"{cap['id']}: drift {drift:+.3f} +/- {drift_err:.3f} deg/min, "
          f"std@100ms = {stats.get(0.1, {}).get('std_deg', float('nan')):.4f} deg")

# Stability vs window length table (first five windows = the report table).
table = []
for cid, r in results.items():
    for w in WINDOWS_S[:5]:
        if w in r["stats"]:
            table.append({"capture": cid, "window_s": w,
                          "n_windows": r["stats"][w]["n_windows"],
                          "mean_deg": r["stats"][w]["mean_deg"],
                          "std_deg": r["stats"][w]["std_deg"]})
df = pd.DataFrame(table)
df


In [ ]:
# phi12(t) for the longest capture (first 60 s shown).
long_id = max(results, key=lambda k: results[k]["phi12"].size)
r = results[long_id]
t = np.arange(r["phi12"].size) / r["fs_phi"]
sel = t <= 60.0
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t[sel], np.rad2deg(r["phi12"][sel]), lw=0.5)
ax.set_xlabel("t [s]"); ax.set_ylabel("phi12 [deg]")
ax.set_title(f"inter-channel phase, {long_id}"
             + (" (SYNTHETIC)" if r["synthetic"] else ""))
fig.tight_layout(); plt.show()


In [ ]:
# Allan deviation (oscillator-style stability curve) and std-vs-window.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for cid, r in results.items():
    label = cid + (" (SYNTHETIC)" if r["synthetic"] else "")
    ax1.loglog(r["taus"], r["adev"], label=label)
    ws = sorted(r["stats"])
    ax2.loglog(ws, [r["stats"][w]["std_deg"] for w in ws], "o-", label=label)
ax1.set_xlabel("tau [s]"); ax1.set_ylabel("ADEV(phi12) [rad]")
ax1.set_title("overlapping Allan deviation"); ax1.legend(fontsize=7)
ax1.axhline(np.deg2rad(THRESHOLD_DEG), color="k", ls="--", lw=0.8)
ax2.set_xlabel("window length [s]"); ax2.set_ylabel("std(phi12) [deg]")
ax2.set_title("phase stability vs window length"); ax2.legend(fontsize=7)
ax2.axhline(THRESHOLD_DEG, color="k", ls="--", lw=0.8)
fig.tight_layout(); plt.show()


## Integer offset and per-channel calibration snapshot

The integer sample offset between channels must be constant across
recordings; DC offset, IQ imbalance and noise-floor difference go into
`data/calibration/pluto_plus.json`.


In [ ]:
cap0 = captures[0]
seg = min(int(cap0["fs"]), cap0["z1"].size, cap0["z2"].size)
lag0 = coherence.integer_sample_offset(cap0["z1"][:seg], cap0["z2"][:seg])
print(f"integer offset (first capture): {lag0} samples")

cal = {
    "capture": cap0["id"],
    "synthetic": cap0["synthetic"],
    "integer_offset_samples": int(lag0),
    "rx1": {**coherence.iq_imbalance_tone(cap0["z1"][:seg]),
            "noise_floor_db": coherence.noise_floor_dbfs(cap0["z1"][:seg])},
    "rx2": {**coherence.iq_imbalance_tone(cap0["z2"][:seg]),
            "noise_floor_db": coherence.noise_floor_dbfs(cap0["z2"][:seg])},
}
# JSON has no complex type: store DC as components.
for ch in ("rx1", "rx2"):
    dc = cal[ch]["dc"]
    cal[ch]["dc"] = {"i": float(dc.real), "q": float(dc.imag)}
coherence.write_calibration(cal, CAL_PATH)
print(f"calibration written to {CAL_PATH}")
cal


## Headline: CPI bound

The deliverable is the largest window with std(phi12) < 10 deg, taken from
the WORST capture (conservative). Reference: DVB-T passive radar typically
uses 10-100 ms CPIs; if the 10 deg bound holds for >= 100 ms, phase P3 is
comfortable. If it does not, the honest number is still the deliverable.


In [ ]:
cpi = {cid: coherence.max_cpi_window(r["stats"], THRESHOLD_DEG)
       for cid, r in results.items()}
for cid, v in cpi.items():
    ms = "n/a" if v is None else f"{1000*v:.0f} ms"
    print(f"{cid}: max window with std < {THRESHOLD_DEG:.0f} deg -> {ms}")

valid = [v for v in cpi.values() if v is not None]
worst_ms = min(valid) * 1000 if valid else None
verdict = (
    "SYNTHETIC placeholder - run at the bench for the real number"
    if any(r["synthetic"] for r in results.values())
    else "MEASURED"
)
print(f"\nCPI bound ({verdict}): "
    f"{worst_ms if worst_ms is not None else 'below 10 ms'} "
    f"(DVB-T reference: 10-100 ms)")
